<a href="https://colab.research.google.com/github/rafi1624/Machine-Learning_18_Rafi-Abyantara/blob/main/JS09/JS09%20-%20Lab%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 10.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.7 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=191c1b2418a3daa12cd3294264b3c01957b1e5fc8a00a1d834b8d1e7a3c2efc5
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679801 sha256=6f3525f3e71eb0c310c2c47eaffb968b8fa8d5ce45dcfead7176a1bb308e849e
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f5

In [4]:
	import numpy as np
	import time
	from annoy import AnnoyIndex
	import faiss
	import hnswlib

	# ===============================
	# Build 1mio rows with 5D
	# ===============================
	n_data = 100_000   # try 100_000 if you have limited memory
	dim = 5
	X = np.random.random((n_data, dim)).astype(np.float32)

	# Query point
	query = np.random.random((1, dim)).astype(np.float32)
	k = 10

	# ===============================
	# 2. Annoy
	# ===============================
	print("=== Annoy ===")
	ann_index = AnnoyIndex(dim, 'euclidean')

	start = time.time()
	for i in range(n_data):
	    ann_index.add_item(i, X[i])
	ann_index.build(10)  # 10 trees
	build_time = time.time() - start

	start = time.time()
	neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
	query_time = time.time() - start

	print("Build time:", build_time, "detik")
	print("Query time:", query_time, "detik")
	print("Neighbors:", neighbors[0][:5], "...")

	# ===============================
	# 3. FAISS (Flat Index)
	# ===============================
	print("\n=== FAISS (IndexFlatL2) ===")
	faiss_index = faiss.IndexFlatL2(dim)

	start = time.time()
	faiss_index.add(X)
	build_time = time.time() - start

	start = time.time()
	distances, indices = faiss_index.search(query, k)
	query_time = time.time() - start

	print("Build time:", build_time, "detik")
	print("Query time:", query_time, "detik")
	print("Neighbors:", indices[0][:5], "...")

	# ===============================
	# 4. HNSW (hnswlib)
	# ===============================
	print("\n=== HNSW (hnswlib) ===")
	hnsw_index = hnswlib.Index(space='l2', dim=dim)

	start = time.time()
	hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
	hnsw_index.add_items(X)
	build_time = time.time() - start

	hnsw_index.set_ef(50)

	start = time.time()
	labels, distances = hnsw_index.knn_query(query, k=k)
	query_time = time.time() - start

	print("Build time:", build_time, "detik")
	print("Query time:", query_time, "detik")
	print("Neighbors:", labels[0][:5], "...")

=== Annoy ===
Build time: 1.606074571609497 detik
Query time: 0.000240325927734375 detik
Neighbors: [85924, 20262, 64902, 38348, 27568] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.0008141994476318359 detik
Query time: 0.0012173652648925781 detik
Neighbors: [85924 20262 64902 38348 27568] ...

=== HNSW (hnswlib) ===
Build time: 12.072110652923584 detik
Query time: 0.00017380714416503906 detik
Neighbors: [85924 20262 64902 38348 27568] ...
